In [1]:
import pandas as pd
import numpy as np
from src.dataset_manager import DatasetManager
from src.metrics_manager import Metrics
# from src.models.negative_binomial import NegativeBinomialPiecewise
from sklearn.preprocessing import RobustScaler

In [ ]:
m_train, m_test = DatasetManager.split_dataset()

Motores para entrenamiento: 140
Motores para prueba: 60


In [ ]:
import statsmodels.api as sm
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
import numpy as np
import warnings

class NegativeBinomialPiecewise(BaseEstimator, RegressorMixin):
    def __init__(self, 
                 alpha: float = 1.0, 
                 clipping_threshold: int = 125,
                 alpha_reg: float = 0.0,
                 l1_ratio: float = 0.5,
                 link_type: str = 'log'): # 'log', 'identity', 'sqrt'
        self.alpha = alpha
        self.clipping_threshold = clipping_threshold
        self.alpha_reg = alpha_reg
        self.l1_ratio = l1_ratio
        self.link_type = link_type
        self.is_fitted_ = False

    def _get_link(self):
        links = {
            'log': sm.families.links.Log(),
            'identity': sm.families.links.Identity(),
            'sqrt': sm.families.links.Sqrt()
        }
        return links.get(self.link_type, sm.families.links.Log())

    def predict(self, X):
        if not getattr(self, 'is_fitted_', False) or self.model_stats_ is None:
            return np.full(X.shape[0], np.nan)
        X = check_array(X)
        X_with_const = sm.add_constant(X, has_constant='add')
        return self.model_stats_.predict(X_with_const)
    
    def fit(self, X, y):
            X, y = check_X_y(X, y, accept_sparse=True)
            y_piecewise = np.minimum(y, self.clipping_threshold)
            X_with_const = sm.add_constant(X, has_constant='add')
            
            try:
                family_nb = sm.families.NegativeBinomial(
                    alpha=self.alpha, 
                    link=self._get_link()
                )
                model = sm.GLM(y_piecewise, X_with_const, family=family_nb)

                if self.alpha_reg > 0:
                    self.model_stats_ = model.fit_regularized(
                        method='elastic_net',
                        alpha=self.alpha_reg,
                        L1_wt=self.l1_ratio,
                        maxiter=500
                    )
                else:
                    self.model_stats_ = model.fit()
                
                self.is_fitted_ = True
                self.fit_success_ = 1  # Marcamos éxito
                
            except Exception as e:
                self.is_fitted_ = False
                self.model_stats_ = None
                self.fit_success_ = 0 
                warnings.warn(f"Iteración fallida: {self.link_type} con alpha_reg {self.alpha_reg}")
                
            return self



In [4]:
def preparar_entrenamiento_grupal(lista_ids: list) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    X_list = []
    y_survival_list = []
    y_count_list = []
    groups_list = []

    for idx in lista_ids:
        # Cargar motor
        df_motor = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
        
        # 1. Identificar si el motor es censurado y su tiempo máximo observado
        es_evento = bool(df_motor['evento'].iloc[0]) # En FD001, o falla o es censurado, no cambia a mitad
        max_tiempo_observado = df_motor['time_in_cycles'].max()
        tiempo_muerte_real = max_tiempo_observado + df_motor['RUL'].iloc[-1]
        
        # 2. X: Sensores y settings
        X_motor = df_motor.drop(columns=['time_in_cycles', 'RUL', 'evento'])
        X_list.append(X_motor)
        
        # 3. Lógica de etiquetas fila por fila
        for _, row in df_motor.iterrows():
            # y_count: Siempre es el RUL real (porque esto es para regresión)
            y_count_list.append(row['RUL'])
            
            # y_survival: Aquí aplicamos la corrección
            if es_evento:
                # Si falló, el horizonte es el tiempo de muerte real
                y_survival_list.append((True, tiempo_muerte_real))
            else:
                # Si es censurado, el horizonte es SOLO lo que vimos (max_tiempo_observado)
                y_survival_list.append((False, max_tiempo_observado))
            
            groups_list.append(idx)

    X = pd.concat(X_list, ignore_index=True)
    y_surv = np.array(y_survival_list, dtype=[('evento', bool), ('tiempo', float)])
    y_count = np.array(y_count_list)
    groups = np.array(groups_list)
    
    return X, y_surv, y_count, groups

# Generar datos para el Grid Search usando tus 140 motores de entrenamiento
X_train, y_surv_train, y_count_train, groups_train = preparar_entrenamiento_grupal(m_train)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import GroupKFold, GridSearchCV

def group_grid_search(param_grid, scoring, n_folds=5):
    # Definimos el scaler
    scaler = RobustScaler()

    # El Pipeline es vital: asegura que el escalado se haga SOLO 
    # con datos de entrenamiento en cada fold (evita Data Leakage)
    base_pipeline = Pipeline([
        ('scaler', scaler),
        ('model', NegativeBinomialPiecewise()) # Usamos nuestra clase como base
    ])

    # Configuramos el GroupKFold para no separar ciclos de un mismo motor
    gkf = GroupKFold(n_splits=n_folds)
    
    return GridSearchCV(
        base_pipeline, 
        param_grid=param_grid, 
        cv=gkf,
        refit='S_score', # Asegúrate de que este nombre esté en tu dict de scoring
        scoring=scoring,
        n_jobs=2, 
        error_score=np.nan,
        return_train_score=True
    )

In [ ]:
import joblib
from tqdm.auto import tqdm
from contextlib import contextmanager
import math

metrics = Metrics.get_metrics()

param_grid = {
    'model__alpha': [0.1, 0.5, 1.0, 1.5],
    'model__link_type': ['log', 'identity', 'sqrt'], 
    'model__alpha_reg': [0.0, 0.05, 0.1, 0.5],            
    'model__l1_ratio': [0.0, 0.5, 0.75, 1.0],              
    'model__clipping_threshold': [115, 120, 125, 130]              
}

# Multiplica el largo de todas las listas en el diccionario
n_combinaciones = math.prod(len(v) for v in param_grid.values())

# 3. Define el total de tareas
n_folds = 5  # El número de divisiones de tu GroupKFold
total_tasks = n_combinaciones * n_folds

print(f"Total de tareas a ejecutar: {total_tasks}")

ggs = group_grid_search(param_grid=param_grid, scoring=metrics, n_folds=5)

@contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager para parchear joblib y mostrar barra de progreso de tqdm."""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

Total de tareas a ejecutar: 3840


/home/jdani/proyects/Premant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import warnings
import os

# 1. Suprimir warnings de Python (UserWarnings, FutureWarnings, etc.)
warnings.filterwarnings("ignore")

# 2. Suprimir warnings específicos de Statsmodels (convergencia y dominio)
from statsmodels.tools.sm_exceptions import ConvergenceWarning, DomainWarning, ValueWarning
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', DomainWarning)
warnings.simplefilter('ignore', ValueWarning)

# 3. Suprimir warnings de Scikit-Learn/Joblib a nivel de sistema (opcional)
# Esto evita que los workers envíen mensajes de error al buffer de salida estándar
os.environ['PYTHONWARNINGS'] = 'ignore'

# --- Tu ejecución aquí ---
print("Ejecutando Grid Search en modo silencioso...")
with tqdm_joblib(tqdm(desc="GGS en progreso", total=total_tasks)) as progress_bar:
    ggs.fit(X_train, y_count_train, groups=groups_train)

Ejecutando Grid Search en modo silencioso...


GGS en progreso: 100%|██████████| 3840/3840 [5:57:39<00:00,  5.59s/it]   


In [ ]:
def obtener_resumen_final(grid_search_object):
    cols_map = {
        'param_model__link_type': 'Link',
        'param_model__alpha' : 'Alpha',
        'param_model__alpha_reg': 'Penalty',
        'param_model__l1_ratio': 'L1_Ratio',
        'param_model__clipping_threshold': 'Threshold',
        'mean_test_C_index': 'C-Index',
        'mean_test_S_score': 'S-Score',
        'mean_test_MAE': 'MAE',
        'mean_test_RMSE': 'RMSE'
    }
    
    df = pd.DataFrame(grid_search_object.cv_results_)
    df = df[list(cols_map.keys())].rename(columns=cols_map)
    
    # Creamos la columna Success basándonos en si el MAE es un número real
    df['Success'] = df['MAE'].apply(lambda x: 0 if np.isnan(x) else 1)
    
    # Rellenamos los NaNs con valores de castigo para que al ordenar 
    # las fallidas queden al final
    df['S-Score'] = df['S-Score'].fillna(999999)
    df['MAE'] = df['MAE'].fillna(999999)
    
    # Ordenar por éxito y luego por S-Score
    df = df.sort_values(by=['Success', 'S-Score'], ascending=[False, True])
    
    return df

# --- USO DESPUÉS DEL FIT ---
res_df = obtener_resumen_final(ggs)
display(res_df.head(15))

,Link,Penalty,L1_Ratio,Threshold,C-Index,S-Score,MAE,RMSE,Success
633,log,0.05,1.00,115,0.853675,7.975524,18.865577,22.674074,1
630,log,0.05,0.75,115,0.855432,8.826172,20.253591,24.507686,1
294,log,0.10,0.75,115,0.858197,8.833283,18.710940,22.938103,1
297,log,0.10,1.00,115,0.858141,8.949517,17.858564,21.855988,1
489,log,0.10,1.00,115,0.851272,8.959525,20.834251,24.276119,1
438,log,0.05,0.75,115,0.857810,9.055530,18.709937,22.975160,1
153,log,0.50,1.00,115,0.849942,9.168318,18.333627,22.233180,1
150,log,0.50,0.75,115,0.848834,9.182501,19.446565,23.605988,1
645,log,0.05,1.00,120,0.846329,9.404571,19.792009,23.859046,1
441,log,0.05,1.00,115,0.856557,9.518486,17.964788,22.007664,1


In [ ]:
param_grid_refined = {
    'model__link_type': ['log'], # Solo el ganador
    'model__alpha': [0.1, 0.25, 0.5], # Zoom a dispersiones bajas si el C-index sube
    'model__alpha_reg': [0.01, 0.03, 0.05, 0.07, 0.1], # Zoom entre 0 y 0.1
    'model__l1_ratio': [0.8, 0.9, 1.0], # Ya vimos que Lasso (1.0) funciona muy bien
    'model__clipping_threshold': [110, 115, 120] # Alrededor del 115 que ganó
}

# Multiplica el largo de todas las listas en el diccionario
n_combinaciones = math.prod(len(v) for v in param_grid_refined.values())

# 3. Define el total de tareas
n_folds = 5  # El número de divisiones de tu GroupKFold
total_tasks = n_combinaciones * n_folds

print(f"Total de tareas a ejecutar: {total_tasks}")

ggs_refined = group_grid_search(param_grid=param_grid_refined, scoring=metrics, n_folds=5)


Total de tareas a ejecutar: 675


In [ ]:
# 1. Suprimir warnings de Python (UserWarnings, FutureWarnings, etc.)
warnings.filterwarnings("ignore")

# 2. Suprimir warnings específicos de Statsmodels (convergencia y dominio)
from statsmodels.tools.sm_exceptions import ConvergenceWarning, DomainWarning, ValueWarning
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', DomainWarning)
warnings.simplefilter('ignore', ValueWarning)

# 3. Suprimir warnings de Scikit-Learn/Joblib a nivel de sistema (opcional)
# Esto evita que los workers envíen mensajes de error al buffer de salida estándar
os.environ['PYTHONWARNINGS'] = 'ignore'

# --- Tu ejecución aquí ---
print("Ejecutando Grid Search en modo silencioso...")
with tqdm_joblib(tqdm(desc="GGS en progreso", total=total_tasks)) as progress_bar:
    ggs_refined.fit(X_train, y_count_train, groups=groups_train)

Ejecutando Grid Search en modo silencioso...


GGS en progreso: 100%|██████████| 675/675 [1:06:35<00:00,  5.92s/it]


In [1]:
def obtener_resumen_final(grid_search_object):
    cols_map = {
        'param_model__link_type': 'Link',
        'param_model__alpha' : 'Alpha',
        'param_model__alpha_reg': 'Penalty',
        'param_model__l1_ratio': 'L1_Ratio',
        'param_model__clipping_threshold': 'Threshold',
        'mean_test_C_index': 'C-Index',
        'mean_test_S_score': 'S-Score',
        'mean_test_MAE': 'MAE',
        'mean_test_RMSE': 'RMSE'
    }
    
    df = pd.DataFrame(grid_search_object.cv_results_)
    df = df[list(cols_map.keys())].rename(columns=cols_map)
    
    # Creamos la columna Success basándonos en si el MAE es un número real
    df['Success'] = df['MAE'].apply(lambda x: 0 if np.isnan(x) else 1)
    
    # Rellenamos los NaNs con valores de castigo para que al ordenar 
    # las fallidas queden al final
    df['S-Score'] = df['S-Score'].fillna(999999)
    df['MAE'] = df['MAE'].fillna(999999)
    
    # Ordenar por éxito y luego por S-Score
    df = df.sort_values(by=['Success', 'S-Score'], ascending=[False, True])
    
    return df

In [2]:
# --- USO DESPUÉS DEL FIT ---
res_df_refined = obtener_resumen_final(ggs_refined)
display(res_df_refined.head(15))

NameError: name 'ggs_refined' is not defined